In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from joblib import Parallel, delayed
from datetime import datetime
import sys
from pathlib import Path
from tqdm import tqdm

import sys
# sys.path.append("../src")
sys.path.append(str(Path.cwd().parent))

# from indicators.indicators import load_symbol, build_indicators
from mdt.mdt import load_symbol
from backtester import engine

import vars

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 40)
pd.set_option('display.max_colwidth', 200)

color_back = "#0d1117"

path_repo = vars.path_repo
path_data = vars.path_data

In [2]:
symbs = vars.symbs

symbs_crypto = [x for x in symbs if "-USD" in x]
symbs_forex = [x for x in symbs if "=X" in x]
symbs_stocks = [x for x in symbs if x not in symbs_crypto + symbs_forex]

print("Total:", len(symbs))
print("Crypto :", len(symbs_crypto), f"({(len(symbs_crypto)/len(symbs))*100:.0f}%)")
print("Forex:", len(symbs_forex), f"({(len(symbs_forex)/len(symbs))*100:.0f}%)")
print("Stocks:", len(symbs_stocks), f"({(len(symbs_stocks)/len(symbs))*100:.0f}%)")

Total: 12
Crypto : 3 (25%)
Forex: 3 (25%)
Stocks: 6 (50%)


In [3]:
symb = "GOOGL"
df = load_symbol(symb, "1h", 400) 


# cols_sign = [col for col in df.columns if col.startswith("sign_")] 
# cols_indic = [col for col in df.columns if not col.startswith("sign_")] 

cols_sign = [
# 'sign_random', 'sign_random2', 
# 'sign_macd_rsi_robust', #'sign_macd_rsi_simple', #'sign_macd_rsi_simple_break', #'sign_macd_rsi_simple_bb_pos', 'sign_macd_rsi_bb', 'sign_macd_rsi_bb_ema', 
# 'sign_adx_crec', 'sign_adx_crec_rsi', #'sign_adx_rsi',
# 'sign_adx_rsi_robust', #'sign_adx_rsi_xtrem', 
'sign_vpvma', 'sign_sinergia', #'sign_rsi_bb_rever', #'sign_ma_macd', 'str_squ_kel_bb', 
# 'sign_gold', 
# 'sign_rsi30', #'sign_rsi20', 'sign_rsi15', 'sign_pullb_ema', 'sign_rsi_div', 'sign_supo_resi', 'sign_sobreextema', 'sign_stochastic', 'sign_cmf', 'sign_elderray', 
'sign_willr', #'sign_donchian', 'sign_liquidsweep', 'sign_macd_div1', 'sign_macd_div3', 'sign_macd_div5', 'sign_random3'
]

cols_sign = [
'sign_random', 'sign_random2', 'sign_macd_rsi_robust', 'sign_macd_rsi_simple', 'sign_macd_rsi_simple_break', 'sign_macd_rsi_simple_bb_pos', 'sign_macd_rsi_bb', 'sign_macd_rsi_bb_ema', 
'sign_adx_crec', 'sign_adx_crec_rsi', 'sign_adx_rsi', 'sign_adx_rsi_robust', 'sign_adx_rsi_xtrem', 'sign_vpvma', 'sign_sinergia', 'sign_rsi_bb_rever', 'sign_ma_macd', 'str_squ_kel_bb', 
'sign_gold', 'sign_rsi30', 'sign_rsi20', 'sign_rsi15', 'sign_pullb_ema', 'sign_rsi_div', 'sign_supo_resi', 'sign_sobreextema', 'sign_stochastic', 'sign_cmf', 'sign_elderray', 
'sign_willr', 'sign_donchian', 'sign_liquidsweep', 'sign_macd_div1', 'sign_macd_div3', 'sign_macd_div5', 'sign_random3'
]

cols_indic = [
'Close', 'High', 'Low', 'Open', 'Volume', 'date', 'EMA20', 'EMA50', 'EMA200', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 'BB_upper', 'BB_mid', 'BB_lower', 'BB_position', 
'STOCH_K', 'STOCH_D', 'ATR', 'OBV', 'OBV_EMA', 'ADX', 'ADX+DI', 'ADX-DI', 'prev26', 'prev52', 'VPVMA', 'VPVMAS', 'VWAP', 'spread', 'vol_ma', 'volume_ma', 'volume_increasing', 
'order_flow', 'volatility', 'CMF', 'PSAR', 'WillR', 'Donchian_high', 'Donchian_low', 'ema50_distance', 'MACD_bearish_div', 'MACD_bullish_div', 'ind_volume_anomaly', 'ind_compress_range', 
'ind_vwap_deviation', 'ind_3ema', 'ind_adx', 'trend_adx',
]

order = sorted(cols_sign)
# df = df[order]

print("Signals   :", len(cols_sign))
print("Indicators:", len(cols_indic))

Signals   : 36
Indicators: 50


In [4]:
df["mes"] = df["date"].dt.strftime("%Y-%m-%d")
_days = len(df["mes"].unique())
aux = df[cols_sign].replace({0:np.nan}).count().reset_index().sort_values(0, ascending = False)
aux["days"] = _days
aux["byday"] = aux[0] / aux["days"]
dels = aux[(aux[0] < 20) | (aux["byday"] > 1.5)]["Price"].tolist()
print(len(dels))
print(dels)

12
['sign_ma_macd', 'sign_cmf', 'sign_adx_crec', 'sign_vpvma', 'sign_sobreextema', 'sign_rsi15', 'sign_macd_rsi_bb_ema', 'sign_adx_rsi', 'sign_macd_rsi_simple_break', 'sign_adx_crec_rsi', 'sign_adx_rsi_xtrem', 'sign_adx_rsi_robust']


In [5]:
print(len(cols_sign))
cols_sign = [x for x in cols_sign if x not in dels]
print(len(cols_sign))

36
24


In [6]:
def calculate_score(resume):
    resume = resume.copy()
    variables = ['return', 'sharpe', 'profit_factor', "calmar", "expectancy"]
    cols_rank = []
    for col in variables:
        resume[f'rank_{col}'] = resume[col].fillna(0).rank(method='min', ascending=False).astype(int)
        cols_rank += [f'rank_{col}']

    resume["score"] = 0.30 * resume["rank_sharpe"] + 0.20 * resume["rank_return"] + 0.20 * resume["rank_profit_factor"] + 0.15 * resume["rank_calmar"] + 0.15 * resume["rank_expectancy"]

    resume.drop(columns = cols_rank, inplace = True)


    resume["score"] = np.where(resume["trades"] < 5, resume["score"] * 0.5, resume["score"])
    resume["score"] = np.where(resume["trades_byday"] >= 4, resume["score"] * 0.5, resume["score"])
    resume["score"] = np.where(resume["profit_factor"] < 1.1, resume["score"] * 0.5, resume["score"])

    # if max_drawdown < -0.30:
    #     score *= 0.5

    return resume


def symbol_performance(df, _cols_sign, _symb, atr_sl = 2,
                                                multiplier_tp=1.5,
                                                use_trailing_stop=False,
                                                use_breakeven=False, # Se activa solo si está activo use_trailing_stop
                                                trailing_atr_mult=1,
                                                trailing_activation_r=2,
                                                breakeven_trigger_r=2):

    params = [atr_sl, multiplier_tp, use_trailing_stop, use_breakeven, trailing_atr_mult, trailing_activation_r, breakeven_trigger_r]

    def _symbol(df, signal, _symb, params): 

        resume, trades, equity = engine(df, signal, _symb,
                                        initial_capital=1000,
                                        risk_per_trade=0.01,
                                        fee_rate=0.0007,
                                        slippage_pct=0.0005,
                                        leverage=100,
                                        atr_sl=params[0],
                                        multiplier_tp=params[1], 
                                        use_trailing_stop=params[2],
                                        use_breakeven=params[3],      # Se activa solo si está activo use_trailing_stop
                                        trailing_atr_mult=params[4],
                                        trailing_activation_r=params[5],
                                        breakeven_trigger_r=params[6])
        
        resume["signal"] = signal
        resume["symbol"] = _symb
        trades["signal"] = signal
        trades["symbol"] = _symb
        equity["signal"] = signal
        equity["symbol"] = _symb

        return resume, trades, equity

    results = Parallel(n_jobs=-1, backend="threading")(delayed(_symbol)(df, signal, _symb, params) for signal in _cols_sign)

    resume =  []
    trades = []
    equity = []

    for _ax1, _ax2, _ax3 in results:
        resume.append(_ax1)
        trades.append(_ax2)
        equity.append(_ax3)
        
    resume = pd.concat(resume).sort_values("capital_final", ascending = False)
    trades = pd.concat(trades)
    equity = pd.concat(equity)

    # print(len(resume))

    tmp = list(set(df["date"].astype(str).str[:10].tolist()))
    total = len(tmp)
    resume["trades_byday"] = resume["trades"] / total

    resume["param.atr_sl"] = atr_sl
    resume["param.multiplier_tp"] = multiplier_tp
    resume["param.use_trailing_stop"] = use_trailing_stop
    resume["param.use_breakeven"] = use_breakeven
    resume["param.trailing_atr_mult"] = trailing_atr_mult
    resume["param.trailing_activation_r"] = trailing_activation_r
    resume["param.breakeven_trigger_r"] = breakeven_trigger_r

    resume = calculate_score(resume)
 
    return resume, trades, equity




In [7]:
resume, trades, equity = symbol_performance(df, cols_sign, symb, atr_sl = 2, multiplier_tp = 1.5, use_trailing_stop = False, use_breakeven = False, trailing_atr_mult = 1,
                                                                 trailing_activation_r = 2.5, breakeven_trigger_r = 3)

In [8]:
resume[['symbol', 'signal', "score", 'capital_final', 'pnl', 'return', 'trades', 'win_rate', 'trades_byday', 'expectancy', 'profit_factor', 'sharpe',
        'n_tp', 'n_sl', 'n_ts', 'n_rev', 'fees']] # 'avg_win', 'avg_loss', 'max_drawdown', 

,symbol,signal,score,capital_final,pnl,return,trades,win_rate,trades_byday,expectancy,profit_factor,sharpe,n_tp,n_sl,n_ts,n_rev,fees
0,GOOGL,sign_rsi20,0.500,981.721525,-18.278475,-0.018278,24,0.416667,0.065753,-0.398072,0.937224,-0.532252,10,14,0,0,17.458854
0,GOOGL,sign_gold,1.075,947.386513,-52.613487,-0.052613,44,0.409091,0.120548,-0.786345,0.880472,-1.084466,18,26,0,0,36.137580
0,GOOGL,sign_macd_rsi_robust,4.675,919.917626,-80.082374,-0.080082,21,0.380952,0.057534,-3.447276,0.584550,-2.384360,8,13,0,0,15.485122
0,GOOGL,sign_macd_rsi_simple_bb_pos,8.575,892.614205,-107.385795,-0.107386,12,0.166667,0.032877,-8.518716,0.202754,-4.563516,2,10,0,0,10.409713
0,GOOGL,sign_macd_rsi_simple,7.450,884.060820,-115.939180,-0.115939,17,0.235294,0.046575,-6.427012,0.315435,-4.120963,4,13,0,0,13.468374
0,GOOGL,sign_rsi_div,4.850,852.622455,-147.377545,-0.147378,44,0.386364,0.120548,-2.953162,0.629857,-3.117106,16,26,0,2,34.940862
0,GOOGL,sign_macd_rsi_bb,3.450,850.981614,-149.018386,-0.149018,62,0.387097,0.169863,-2.053519,0.711332,-2.542528,23,38,0,1,43.493588
0,GOOGL,sign_pullb_ema,2.825,836.496869,-163.503131,-0.163503,144,0.361111,0.394521,-0.777601,0.854707,-2.165860,50,33,0,61,108.373494
0,GOOGL,sign_sinergia,4.750,825.409957,-174.590043,-0.174590,74,0.351351,0.202740,-1.898757,0.713575,-3.536952,26,43,0,5,58.504651
0,GOOGL,sign_macd_div3,5.600,809.010241,-190.989759,-0.190990,59,0.406780,0.161644,-2.826782,0.641003,-3.320353,22,35,0,2,46.672003


### Tuning

In [9]:
# Calculate train and test
df = load_symbol(symb, "1h", 600) 

In [10]:
porc_train = 0.7 
porc_eval = 0.15

df["mes"] = df["date"].dt.strftime("%Y-%m-%d")
ax = df.drop_duplicates("mes")
c = int(len(ax)*porc_train)
c2 = int(len(ax)*(porc_eval+porc_train))

dates_train = [ax.iloc[:c]["mes"].tolist()[0], ax.iloc[:c]["mes"].tolist()[-1]]
dates_eval = [ax.iloc[c:]["mes"].tolist()[0], ax.iloc[:c2]["mes"].tolist()[-1]]
dates_oos = [ax.iloc[c2:]["mes"].tolist()[0], ax.iloc[c2:]["mes"].tolist()[-1]]

df_train = df[(df["date"] >= dates_train[0]) & (df["date"] <= dates_train[1])]
df_eval = df[(df["date"] >= dates_eval[0]) & (df["date"] <= dates_eval[1])]
df_oos = df[(df["date"] >= dates_oos[0]) & (df["date"] <= dates_oos[1])]

print("Train:", dates_train)
print("Eval :", dates_eval)
print("OOS  :", dates_oos, "\n")

print(f"{len(df_train)/len(df)*100:.1f}%")
print(f"{len(df_eval)/len(df)*100:.1f}%")
print(f"{len(df_oos)/len(df)*100:.1f}%")

Train: ['2024-07-16', '2026-02-09']
Eval : ['2026-02-10', '2026-06-11']
OOS  : ['2026-06-12', '2026-10-13'] 

69.5%
15.0%
15.0%


In [11]:
"""
6. Qué optimizaría primero en tu caso No optimizaría todos los parámetros de una sola vez. Lo haría en tres etapas:
1. SL y TP: atr_sl, multiplier_tp,
2. Gestión de beneficios: trailing stop, multiplicador ATR y activación.
3. Breakeven: activación y compatibilidad con trailing.


Parámetro	Valores iniciales
atr_sl	1.0, 1.5, 2.0, 2.5, 3.0
multiplier_tp	1.0, 1.5, 2.0, 2.5, 3.0
use_trailing_stop	False, True
trailing_atr_mult	1.0, 1.5, 2.0
trailing_activation_r	1.0, 1.5, 2.0, 2.5, 3.0
use_breakeven	False, True
breakeven_trigger_r	1.0, 1.5, 2.0, 2.5, 3.0
"""

'\n6. Qué optimizaría primero en tu caso No optimizaría todos los parámetros de una sola vez. Lo haría en tres etapas:\n1. SL y TP: atr_sl, multiplier_tp,\n2. Gestión de beneficios: trailing stop, multiplicador ATR y activación.\n3. Breakeven: activación y compatibilidad con trailing.\n\n\nParámetro\tValores iniciales\natr_sl\t1.0, 1.5, 2.0, 2.5, 3.0\nmultiplier_tp\t1.0, 1.5, 2.0, 2.5, 3.0\nuse_trailing_stop\tFalse, True\ntrailing_atr_mult\t1.0, 1.5, 2.0\ntrailing_activation_r\t1.0, 1.5, 2.0, 2.5, 3.0\nuse_breakeven\tFalse, True\nbreakeven_trigger_r\t1.0, 1.5, 2.0, 2.5, 3.0\n'

In [12]:
def build_param_grid():
    grid = []
    for atr_sl in [1, 1.5, 2, 2.5]:
        for tp in [1, 1.5, 2, 2.5]:
            for use_trailing_stop in [True, False]:
                for use_breakeven in [True, False]:
                    for trailing_atr_mult in [1]:
                        for trailing_activation_r in [2]:
                            for breakeven_trigger_r in [2]:
                                
                                if use_breakeven and not use_trailing_stop: # Evita activar dos mecanismos de gestión en configuraciones que todavía no quieres explorar.
                                    continue

                                # if not use_trailing_stop:
                                #     trailing_atr_mult = 0
                                #     trailing_activation_r = 0
                                #     breakeven_trigger_r = 0

                                else:
                                    params = {
                                        "atr_sl": atr_sl,
                                        "multiplier_tp": tp,
                                        "use_trailing_stop": use_trailing_stop,
                                        "use_breakeven": use_breakeven,
                                        "trailing_atr_mult": trailing_atr_mult,
                                        "trailing_activation_r": trailing_activation_r,
                                        "breakeven_trigger_r": breakeven_trigger_r 
                                    }

                                    grid.append(params)
    return grid

param_grid = build_param_grid()
param_grid = pd.DataFrame(param_grid).drop_duplicates()

print("Setups:", len(param_grid))
param_grid.head(10)

Setups: 48


,atr_sl,multiplier_tp,use_trailing_stop,use_breakeven,trailing_atr_mult,trailing_activation_r,breakeven_trigger_r
0,1.0,1.0,True,True,1,2,2
1,1.0,1.0,True,False,1,2,2
2,1.0,1.0,False,False,1,2,2
3,1.0,1.5,True,True,1,2,2
4,1.0,1.5,True,False,1,2,2
5,1.0,1.5,False,False,1,2,2
6,1.0,2.0,True,True,1,2,2
7,1.0,2.0,True,False,1,2,2
8,1.0,2.0,False,False,1,2,2
9,1.0,2.5,True,True,1,2,2


In [13]:
cols_param = ['param.atr_sl', 'param.multiplier_tp', 'param.use_trailing_stop', 'param.use_breakeven', 'param.trailing_atr_mult', 'param.trailing_activation_r', 'param.breakeven_trigger_r']

In [14]:
def tunner_sl_tp(df, cols_sign, symb, param_grid):
    df = df.copy()
    resume_tun = []
    chunk_size = 50
    for inicio in tqdm(range(0, len(param_grid), chunk_size)):
        chunk = param_grid.iloc[inicio:inicio + chunk_size]
        resultados = Parallel(n_jobs=-1, backend="loky")(
            delayed(symbol_performance)(
                df,
                cols_sign,
                symb,
                atr_sl=row["atr_sl"],
                multiplier_tp=row["multiplier_tp"],
                use_trailing_stop=row["use_trailing_stop"],
                use_breakeven=row["use_breakeven"],
                trailing_atr_mult=row["trailing_atr_mult"],
                trailing_activation_r=row["trailing_activation_r"],
                breakeven_trigger_r=row["breakeven_trigger_r"]
            )
            for _, row in chunk.iterrows())
        resume_tun.extend([resultado[0] for resultado in resultados])


    # for i in tqdm(range(0, len(param_grid))):
    #     row = param_grid.iloc[i]
    #     _resume, _trades, _equity = symbol_performance(
    #                                                 df,
    #                                                 cols_sign,
    #                                                 symb,
    #                                                 atr_sl=row["atr_sl"],
    #                                                 multiplier_tp=row["multiplier_tp"],
    #                                                 use_trailing_stop=row["use_trailing_stop"],
    #                                                 use_breakeven=row["use_breakeven"],
    #                                                 trailing_atr_mult=row["trailing_atr_mult"],
    #                                                 trailing_activation_r=row["trailing_activation_r"],
    #                                                 breakeven_trigger_r=row["breakeven_trigger_r"])
    #     resume_tun.append(_resume)
        

    resume_tun = pd.concat(resume_tun, ignore_index=True)
    resume_tun.drop_duplicates(['symbol', 'signal', 'capital_final', 'pnl', 'return', 'trades', 'win_rate', 'expectancy', 'avg_return', 'max_drawdown', 'sharpe'], inplace = True)
    resume_tun = calculate_score(resume_tun)
    resume_tun = resume_tun[["score"] + [col for col in resume_tun.columns if col != "score"]].sort_values("score", ascending=True)
    return resume_tun

In [15]:
k_top = 3
cols_del = ["capital", "avg_win", "avg_loss", "median_return", "sortino","cagr", "calmar", "avg_trade", "profit_factor", "avg_bars",
            "max_loss_streak", "max_win_streak", "recovery_factor", "var_95", "cvar_95", "exposure", "open_trade"]

In [ ]:
# 1. Tuning in train
# resume_train = tunner_sl_tp(df_train, cols_sign, symb, param_grid)

resume_train = []
for _sign in cols_sign:
    print(_sign)
    _ = tunner_sl_tp(df_train, [_sign], symb, param_grid)
    resume_train.append(_)
resume_train = pd.concat(resume_train)

tmp = []
for _sign in cols_sign:
    _ = resume_train[resume_train["signal"] == _sign].sort_values(["signal", "score"], ascending=True).head(k_top)
    tmp.append(_)
resume_train = pd.concat(tmp)
resume_train["dataset"] = "train"
resume_train.drop(columns=cols_del).head(15)

sign_random


  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
param_grid_eval = resume_train[["signal"] + [x for x in resume_train if "param" in x]]
param_grid_eval.columns = [x.replace("param.","") for x in param_grid_eval]
param_grid_eval

In [ ]:
# # 2. Testing in evaluation
# resume_eval = tunner_sl_tp(df_eval, cols_sign, symb, param_grid_eval)
# resume_eval["dataset"] = "eval"
# resume_eval.drop(columns=cols_del).head(15)

In [ ]:
# # 3. Testing in OOS
# resume_oos = tunner_sl_tp(df_oos, cols_sign, symb, param_grid_eval)
# resume_oos["dataset"] = "oos"
# resume_oos.drop(columns=cols_del).head(15)

In [ ]:
# resume_glob = pd.concat([resume_train, resume_eval, resume_oos])
# resume_glob["key"] = resume_glob[cols_param].astype(str).agg("".join, axis = 1)
# resume_glob = resume_glob[["dataset", "key"] + [col for col in resume_glob.columns if col not in ["dataset", "key"]]].sort_values(cols_param, ascending=True)
# resume_glob.drop(columns=cols_del).head(15)

In [ ]:
# ph = vars.path_data + f"{symb}/"
# resume_glob.to_parquet(ph)

## Testing

In [ ]:
# 2. Testing in evaluation
resume_eval = []
for _sign in cols_sign:
    _params = param_grid_eval[param_grid_eval["signal"] == _sign]
    _ = tunner_sl_tp(df_eval, [_sign], symb, _params)
    resume_eval.append(_)
resume_eval = pd.concat(resume_eval)
resume_eval["dataset"] = "eval"
resume_eval.drop(columns=cols_del).head(15)

In [ ]:
# 3. Testing in OOS
resume_oos = []
for _sign in cols_sign:
    _params = param_grid_eval[param_grid_eval["signal"] == _sign]
    _ = tunner_sl_tp(df_oos, [_sign], symb, _params)
    resume_oos.append(_)
resume_oos = pd.concat(resume_oos)
resume_oos["dataset"] = "eval"
resume_oos.drop(columns=cols_del).head(15)

In [ ]:
resume_oos["num"] = 1
resume_oos["counts"] = resume_oos.groupby(["signal"])["num"].transform("sum")

In [ ]:
resume_oos["counts"].value_counts()

In [ ]:
# Join
resume_train["key"] = resume_train[cols_param].astype(str).agg(" ".join, axis = 1).astype(str)
resume_eval["key"] = resume_eval[cols_param].astype(str).agg(" ".join, axis = 1).astype(str)
resume_oos["key"] = resume_oos[cols_param].astype(str).agg(" ".join, axis = 1).astype(str)

keys = ["signal", "key"]
cols = ["pnl", "trades", "win_rate", "profit_factor", "sharpe"]
metr = pd.merge(resume_train[keys + cols], resume_eval[keys + cols], on = keys, how = "left", suffixes = ("_tra", "_eva"))

_resume_oos = resume_oos.copy()[keys + cols].rename(columns = {x:x + "_oos" for x in cols})
metr = pd.merge(metr, _resume_oos, on = keys, how = "left")

for i in cols:
    metr["min_" + i] = metr[[f"{i}_tra", f"{i}_eva", f"{i}_oos"]].min(axis = 1)
metr = metr[(metr["min_pnl"] > 0) & (metr[["pnl_tra", "pnl_eva", "pnl_oos"]].count(axis = 1) == 3)].sort_values("min", ascending = False)

cols = ["signal", "key"] + sorted([x for x in metr if x not in ["signal", "symb"]])
metr = metr[cols]

metr.head(20)

In [ ]:
resume_eval[resume_eval["signal"] == "sign_macd_rsi_robust"].drop(columns = "key") 

In [ ]:
param_grid_eval[param_grid_eval["signal"] == "sign_sinergia"]

In [ ]:
metr["symbol"] = symb
metr.to_parquet(path_data + f"tuning/symbs/{symb}.parquet")

In [ ]:
qwe

## Graph

In [ ]:
if "reason" in trades.columns:

    exit_analysis = (
        trades.groupby("reason")
        .agg(
            operations=("return", "size"),
            total_pnl=("return", "sum"),
            avg_pnl=("return", "mean"),
            win_rate=("return", lambda x: (x > 0).mean()),
            median_pnl=("return", "median")
        )
        .sort_values("total_pnl", ascending=False)
    )

    display(exit_analysis)

    exit_analysis["total_pnl"].plot(
        kind="bar",
        figsize=(12, 4)
    )

    plt.title("PnL acumulado por motivo de salida")
    plt.xlabel("Motivo de salida")
    plt.ylabel("PnL total")
    plt.xticks(rotation=45)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt


dfpl = equity[equity["signal"] == "sign_rsi30"]
dfpl = dfpl.sort_values("time").copy()

dfpl["equity_peak"] = dfpl["equity"].cummax()

dfpl["drawdown"] = (
    dfpl["equity"] / dfpl["equity_peak"] - 1
)

fig, ax = plt.subplots(figsize=(14, 3))

ax.plot(dfpl["time"], dfpl["equity"])

ax.set_title("Equity Curve")
ax.set_xlabel("Fecha")
ax.set_ylabel("Patrimonio ($)")
ax.grid(True)
ax.axhline(1000, linestyle = "--", color = "red")

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(14, 3))

ax.plot(dfpl["time"], dfpl["drawdown"] * 100)

ax.set_title("Drawdown")
ax.set_xlabel("Fecha")
ax.set_ylabel("Drawdown (%)")
ax.grid(True)
ax.axhline(0, linestyle = "--", color = "red")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt


dfpl = equity
dfpl = dfpl.sort_values("time").copy()

dfpl["equity_peak"] = dfpl["equity"].cummax()

dfpl["drawdown"] = (
    dfpl["equity"] / dfpl["equity_peak"] - 1
)

fig, ax = plt.subplots(figsize=(14, 6.5))

for sign in cols_sign:
    tmp = dfpl[dfpl["signal"] == sign]
    ax.plot(tmp["time"], tmp["equity"], label = sign)

ax.set_title("Equity Curve")
ax.set_xlabel("Fecha")
ax.set_ylabel("Patrimonio ($)")
ax.grid(True)
ax.axhline(1000, linestyle = "--", color = "red")

plt.legend()
plt.tight_layout()
plt.show()